# Retail Customer Profitability & Churn Risk Analysis (RFM-Based)

This project analyzes revenue concentration, profit dependency, and churn risk using structured Business Analytics techniques.

## 1. Data Preparation

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('data/Superstore.csv', encoding='latin1')df.columns = df.columns.str.strip()
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Profit_Margin'] = df['Profit'] / df['Sales']
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit_Margin
0,1,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,0.1600
1,2,CA-2016-152156,2016-11-08,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,0.3000
2,3,CA-2016-138688,2016-06-12,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,0.4700
3,4,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,-0.4000
4,5,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,0.1125


## 2. Key Performance Indicators (KPI)

In [3]:
total_revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
overall_margin = total_profit / total_revenue

print('Total Revenue:', round(total_revenue,2))
print('Total Profit:', round(total_profit,2))
print('Overall Profit Margin:', round(overall_margin,4))

Total Revenue: 2297200.86
Total Profit: 286397.02
Overall Profit Margin: 0.1247


## 3. Revenue Concentration Analysis

In [5]:
customer_sales = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False)
top_20_count = int(len(customer_sales) * 0.20)
top_20_revenue = customer_sales.head(top_20_count).sum()
revenue_concentration = (top_20_revenue / total_revenue) * 100

print('Top 20% Customers contribute:', round(revenue_concentration,2), '% of revenue')

Top 20% Customers contribute: 47.96 % of revenue


## 4. Profit Concentration Analysis

In [7]:
customer_profit = df.groupby('Customer Name')['Profit'].sum().sort_values(ascending=False)
top_20_profit = customer_profit.head(top_20_count).sum()
profit_concentration = (top_20_profit / total_profit) * 100

print('Top 20% Customers contribute:', round(profit_concentration,2), '% of profit')

Top 20% Customers contribute: 81.43 % of profit


## 5. Profit Risk Simulation

In [8]:
top_10_percent_of_top = int(top_20_count * 0.10)
lost_profit = customer_profit.head(top_10_percent_of_top).sum()
risk_percentage = (lost_profit / total_profit) * 100

print('If 10% of top customers churn, profit drops by:', round(risk_percentage,2), '%')

If 10% of top customers churn, profit drops by: 22.15 %


## 6. RFM Segmentation

In [11]:
latest_date = df['Order Date'].max()

rfm = df.groupby('Customer Name').agg({
    'Order Date': lambda x: (latest_date - x.max()).days,
    'Order ID': 'nunique',
    'Sales': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
Customer Name,,,
Aaron Bergman,415,3,886.156
Aaron Hawkins,12,7,1744.700
Aaron Smayling,88,7,3050.692
Adam Bellavance,54,8,7755.620
Adam Hart,34,10,3250.337


## 7. High-Value Churn Risk Assessment

In [13]:
recency_threshold = rfm['Recency'].quantile(0.75)

high_risk = rfm[(rfm['Recency'] > recency_threshold) &
                (rfm['Monetary'] > rfm['Monetary'].mean())]

high_risk_names = high_risk.index
profit_at_risk = df[df['Customer Name'].isin(high_risk_names)]['Profit'].sum()
strict_risk_percentage = (profit_at_risk / total_profit) * 100

print('Strict churn profit risk %:', round(strict_risk_percentage,2), '%')

Strict churn profit risk %: 17.64 %


## 8. Executive Summary

- Revenue is moderately concentrated among top 20% customers.
- Profitability is highly concentrated within high-value segments.
- Approximately 17–18% of total profit is exposed under strict churn definition.

### Strategic Recommendations
- Implement targeted retention for high-value inactive customers.
- Use personalized engagement rather than blanket discounting.
- Evaluate ROI before launching retention campaigns.